# 01 - Teacher & Student baseline

In [2]:
import sys; sys.path.append("..")
%load_ext autoreload
%autoreload 2

import torch
import matplotlib.pyplot as plt
from src.data import get_dataloaders
from src.models import build_model
from src.metrics import count_parameters, model_size_mb, save_result
from src.train import get_device, init_weights, fit, evaluate, overfit_one_batch

device = get_device()
torch.backends.cudnn.benchmark = True

train_loader, val_loader, test_loader = get_dataloaders("../data", batch_size=128)

Device: cpu


In [14]:
# Sanity check: overfit su un solo batch (train) per verificare che la loss scenda  (e il training quindi funzioni)
for name in ["teacher", "student"]:
    torch.manual_seed(0)
    m = build_model(name)
    init_weights(m)
    print(f"--- {name} ---")
    overfit_one_batch(m, train_loader, device)

--- teacher ---
step   0 | loss 2.5312
step  20 | loss 0.0962
step  40 | loss 0.0061
step  60 | loss 0.0045
step  80 | loss 0.0004
step  99 | loss 0.0004
--- student ---
step   0 | loss 2.3087
step  20 | loss 1.8431
step  40 | loss 0.6013
step  60 | loss 0.1195
step  80 | loss 0.0624
step  99 | loss 0.0282


In [ ]:
# Baseline training: traina il modello per x epoche, salva il miglior checkpoint sul validation set e registra i risultati in checkpoints e results.json
# funzioni helper per evitare replicazione codice
def run_baseline(name, epochs=30):
    torch.manual_seed(0)
    model = build_model(name)
    init_weights(model)
    model.to(device)

    history, train_time = fit(model, train_loader, val_loader, device, epochs=epochs,
                              ckpt_path=f"../checkpoints/{name}_baseline.pt")

    _, val_acc = evaluate(model, val_loader, device)
    _, test_acc = evaluate(model, test_loader, device)
    c = count_parameters(model)
    print(f"\n{name}: val {val_acc * 100:.2f}% | test {test_acc * 100:.2f}% | "
          f"{c['total']:,} params | {model_size_mb(model):.2f} MB | {train_time / 60:.1f} min")

    save_result(f"{name}_baseline", val_acc=val_acc, test_acc=test_acc,
                params=c["total"], size_mb=model_size_mb(model),
                train_time_s=train_time, epochs=epochs)
    return model, history

In [11]:
# Addestramento del teacher
teacher, h_teacher = run_baseline("teacher",epochs=30)

ep   1/30 | train loss 2.238 acc 16.01 | val loss 1.982 acc 20.74 (best 20.74) | lr 0.0500
ep   2/30 | train loss 1.873 acc 26.58 | val loss 1.643 acc 35.14 (best 35.14) | lr 0.0499
ep   3/30 | train loss 1.564 acc 42.75 | val loss 1.616 acc 47.98 (best 47.98) | lr 0.0495
ep   4/30 | train loss 1.250 acc 56.49 | val loss 1.364 acc 54.40 (best 54.40) | lr 0.0488
ep   5/30 | train loss 1.032 acc 64.78 | val loss 1.030 acc 65.18 (best 65.18) | lr 0.0478
ep   6/30 | train loss 0.893 acc 69.82 | val loss 0.845 acc 71.14 (best 71.14) | lr 0.0467
ep   7/30 | train loss 0.799 acc 72.95 | val loss 0.896 acc 70.62 (best 71.14) | lr 0.0452
ep   8/30 | train loss 0.729 acc 75.76 | val loss 0.734 acc 74.32 (best 74.32) | lr 0.0436
ep   9/30 | train loss 0.665 acc 77.88 | val loss 0.644 acc 77.66 (best 77.66) | lr 0.0417
ep  10/30 | train loss 0.609 acc 79.77 | val loss 0.605 acc 78.92 (best 78.92) | lr 0.0397
ep  11/30 | train loss 0.569 acc 81.10 | val loss 0.608 acc 79.08 (best 79.08) | lr 0.0375

In [15]:
# Addestramento dello student
student, h_student = run_baseline("student",epochs=30)

ep   1/30 | train loss 1.787 acc 32.53 | val loss 1.569 acc 40.72 (best 40.72) | lr 0.0500
ep   2/30 | train loss 1.519 acc 44.85 | val loss 1.537 acc 44.14 (best 44.14) | lr 0.0499
ep   3/30 | train loss 1.392 acc 50.28 | val loss 1.351 acc 49.44 (best 49.44) | lr 0.0495
ep   4/30 | train loss 1.296 acc 54.29 | val loss 1.238 acc 55.56 (best 55.56) | lr 0.0488
ep   5/30 | train loss 1.239 acc 56.69 | val loss 1.244 acc 55.42 (best 55.56) | lr 0.0478
ep   6/30 | train loss 1.194 acc 58.11 | val loss 1.160 acc 58.30 (best 58.30) | lr 0.0467
ep   7/30 | train loss 1.158 acc 59.37 | val loss 1.237 acc 57.16 (best 58.30) | lr 0.0452
ep   8/30 | train loss 1.127 acc 60.61 | val loss 1.079 acc 61.78 (best 61.78) | lr 0.0436
ep   9/30 | train loss 1.095 acc 62.09 | val loss 1.016 acc 63.52 (best 63.52) | lr 0.0417
ep  10/30 | train loss 1.076 acc 62.88 | val loss 1.067 acc 62.68 (best 63.52) | lr 0.0397
ep  11/30 | train loss 1.049 acc 63.74 | val loss 1.142 acc 59.08 (best 63.52) | lr 0.0375

In [3]:
from src.metrics import profile_model, update_result

# Carica i modelli addestrati e profila le loro prestazioni (MACs, parametri, dimensione, latenza) [salva in results.json]
for name in ["teacher", "student"]:
    model = build_model(name)
    model.load_state_dict(torch.load(f"../checkpoints/{name}_baseline.pt", map_location="cpu"))
    prof = profile_model(model, dtype="fp32")
    update_result(f"{name}_baseline", **prof)
    print(name, {k: round(v, 3) if isinstance(v, float) else v for k, v in prof.items()})

teacher {'params': 15530762, 'size_mb': 62.165, 'macs': 215689216, 'energy_uj': 992.17, 'peak_act_kb': 327.68, 'latency_ms_median': 8.303, 'latency_ms_std': 0.215}
student {'params': 110481, 'size_mb': 0.462, 'macs': 1625629, 'energy_uj': 7.478, 'peak_act_kb': 32.768, 'latency_ms_median': 0.719, 'latency_ms_std': 0.106}
